# RAG Arena — Corpus Exploration & Chunk Quality Inspection

This notebook verifies chunking quality before any retrieval index is built.
Inspection is mandatory — chunking is the controlled variable in this study.

In [ ]:
import sys
import os
sys.path.insert(0, "..")

import json
import glob
import tiktoken
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

from src.ingestion.chunker import chunk_document, detect_section

CHUNKS_DIR = "../data/chunks"
ENCODING = tiktoken.get_encoding("cl100k_base")

In [ ]:
all_chunks = []
chunk_files = glob.glob(os.path.join(CHUNKS_DIR, "*_chunks.json"))

for path in sorted(chunk_files):
    with open(path) as f:
        chunks = json.load(f)
    all_chunks.extend(chunks)
    print(f"{os.path.basename(path)}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

In [ ]:
token_lengths = [len(ENCODING.encode(c["text"])) for c in all_chunks]

print(f"Min tokens:    {min(token_lengths)}")
print(f"Max tokens:    {max(token_lengths)}")
print(f"Mean tokens:   {sum(token_lengths)/len(token_lengths):.1f}")
print(f"Chunks > 512:  {sum(1 for t in token_lengths if t > 512)}")
print(f"Chunks < 50:   {sum(1 for t in token_lengths if t < 50)}")

In [ ]:
bad_endings = []
sentence_endings = (".", "?", "!", '"', ")", "}", ":")

for chunk in all_chunks:
    text = chunk["text"].strip()
    if text and not text.endswith(sentence_endings):
        bad_endings.append({"id": chunk["id"], "tail": text[-80:]})

print(f"Chunks NOT ending on sentence boundary: {len(bad_endings)}")
if bad_endings:
    print("\nSamples:")
    for b in bad_endings[:5]:
        print(f"  {b['id']}: ...{b['tail']}")

In [ ]:
sections = [c["section"] for c in all_chunks]
section_counts = Counter(s for s in sections if s)

print(f"Chunks with section header: {sum(1 for s in sections if s)} / {len(sections)}")
print(f"\nTop sections:")
for section, count in section_counts.most_common(10):
    print(f"  {count:4d}  {section}")

In [ ]:
import random
random.seed(42)

sample = random.sample(all_chunks, min(3, len(all_chunks)))
for i, chunk in enumerate(sample):
    print(f"=== Sample {i+1} ===")
    print(f"ID:      {chunk['id']}")
    print(f"Page:    {chunk['page']}")
    print(f"Section: {chunk['section']}")
    print(f"Tokens:  {len(ENCODING.encode(chunk['text']))}")
    print(f"Text:    {chunk['text'][:300]}...")
    print()

## Inspection checklist
- [ ] No chunks > 512 tokens (unless single-sentence edge case)
- [ ] No chunks < 50 tokens (near-empty pages filtered)
- [ ] Bad sentence endings < 1% of total chunks
- [ ] Section headers detected on > 10% of chunks
- [ ] Random samples look like coherent regulatory text

If any box is unchecked, investigate before building retrieval indexes.